# Lecture 7: Train Linear, Ridge, Lasso, and Elastic Net Models

### Short, simple, self-study notes

This lesson trains four regression models on the prepared Algerian Forest Fires data and compares their predictions.

**Main idea:** Linear Regression is a baseline. Ridge, Lasso, and Elastic Net add regularization to control coefficient size and may improve generalization.

## 1. Models we will compare

| Model | What it does |
|---|---|
| Linear Regression | Fits prediction error only; useful baseline |
| Ridge | L2 penalty; shrinks coefficients but usually keeps all features |
| Lasso | L1 penalty; can shrink some coefficients exactly to zero |
| Elastic Net | Mixes L1 and L2; can shrink coefficients and select features |

All four models must use the same train/test split so the comparison is fair.

## 2. Metrics in simple English

- **MAE (Mean Absolute Error):** average prediction mistake in FWI units. Smaller is better.
- **R² score:** how much of the target variation the model explains. Closer to 1 is generally better on unseen test data.
- **Train R² versus test R²:** a very high train score but much lower test score can suggest overfitting.

A lower test score for a regularized model does not automatically mean it is worse. It may be using a simpler model that generalizes better on new data. Cross-validation is the proper way to choose hyperparameters such as alpha.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_path = Path("Ridge Lassso Elastic Regression Practicals") / "Algerian_forest_fires_cleaned_dataset.csv"
df = pd.read_csv(data_path)
df["Classes"] = df["Classes"].str.strip().str.lower().map({"not fire": 0, "fire": 1})

X = df.drop(columns=["day", "month", "year", "FWI"])
y = df["FWI"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Number of input features:", X_train.shape[1])

## 3. Fit the four models

The alpha values below are learning examples, not final best settings. Lasso, Ridge, and Elastic Net need scaling because their penalties depend on coefficient size.

Later, cross-validation will search for better alpha and L1/L2 mixing values.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1, max_iter=20_000),
    "Elastic Net": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20_000),
}

results = []
predictions = {}
coefficients = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_prediction = model.predict(X_train_scaled)
    test_prediction = model.predict(X_test_scaled)

    predictions[name] = test_prediction
    coefficients[name] = model.coef_
    results.append({
        "model": name,
        "train_R2": r2_score(y_train, train_prediction),
        "test_R2": r2_score(y_test, test_prediction),
        "test_MAE": mean_absolute_error(y_test, test_prediction),
        "zero_coefficients": int(np.sum(np.isclose(model.coef_, 0))),
    })

results_table = pd.DataFrame(results).set_index("model").round(3)
results_table

### How to read the results table

- Compare test MAE: lower is better.
- Compare test R²: higher is better.
- Compare train R² and test R²: a large gap can be a warning sign for overfitting.
- Zero coefficients show feature selection. Lasso may create zeros; Ridge generally does not.

One test split is useful for practice, but it is not enough to choose a final alpha. Use cross-validation in the next lesson.

## 4. Visual: actual FWI versus predicted FWI

Each dot is one test record. The dashed diagonal line is perfect prediction.

- Dots close to the line mean predictions are close to real values.
- Dots far from the line are larger errors.
- This visual helps us see patterns that one score alone may hide.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True, sharey=True)
all_values = np.concatenate([y_test.to_numpy(), *predictions.values()])
line_min, line_max = all_values.min(), all_values.max()

for axis, (name, prediction) in zip(axes.ravel(), predictions.items()):
    axis.scatter(y_test, prediction, alpha=0.75, color="#457b9d")
    axis.plot([line_min, line_max], [line_min, line_max], "--", color="#e76f51")
    axis.set_title(name)
    axis.set_xlabel("Actual FWI")
    axis.set_ylabel("Predicted FWI")
    axis.grid(alpha=0.2)

fig.suptitle("Test predictions: closer to the diagonal is better", weight="bold")
fig.tight_layout()
plt.show()

## 5. Visual: how regularization changes coefficients

Because inputs were standardized, coefficient sizes can be compared fairly. Ridge makes weights smaller. Lasso may set some weights to zero. Elastic Net can do both.

In [ ]:
coefficient_table = pd.DataFrame(coefficients, index=X.columns)
coefficient_table.plot(kind="bar", figsize=(14, 5), width=0.8)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Coefficient comparison after feature scaling", weight="bold")
plt.xlabel("Feature")
plt.ylabel("Coefficient")
plt.xticks(rotation=60, ha="right")
plt.legend(title="Model")
plt.tight_layout()
plt.show()

coefficient_table.round(3)

## 6. Final revision card

- Start with Linear Regression as a baseline.
- Ridge uses L2 regularization and usually keeps all features.
- Lasso uses L1 regularization and can make some coefficients zero.
- Elastic Net combines L1 and L2 penalties.
- Use MAE and test R² to compare regression predictions.
- Do not call a model overfit only because another model has a different test score; compare train/test gaps and cross-validation results.
- Alpha and Elastic Net l1_ratio are hyperparameters that should be tuned with cross-validation.

### One-line interview answer

**I compare Linear Regression with Ridge, Lasso, and Elastic Net using the same scaled train/test split, then use cross-validation to tune regularization hyperparameters fairly.**